### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama

model = ChatOllama(model = "llama3.1:8b", profile={"max_input_tokens": 128000})

### Messagebased summarization
agent = create_agent(
    model = model,
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = model,
            trigger = ("messages",10),
            keep = ("messages",4)
        )
    ]
)

### Run with thread id

In [ ]:
from langchain_core.runnables import RunnableConfig

config:RunnableConfig = {"configurable": {"thread_id": "test-1"}}

In [ ]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke(input = {"messages":[HumanMessage(content=q)]}, config = config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='64d2dffb-60ed-4e36-bb31-19a8e10aaaef'), AIMessage(content='2+2 = 4', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-08-27T16:29:38.8940457Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3046310800, 'load_duration': 2830402400, 'prompt_eval_count': 17, 'prompt_eval_duration': 70658000, 'eval_count': 7, 'eval_duration': 132690000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--01a0440e-30e5-7843-adb8-aa694ca8735a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 7, 'total_tokens': 24})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='64d2dffb-60ed-4e36-bb31-19a8e10aaaef'), AIMessage(content='2+2 = 4', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_a

## Token Size

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig 
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model = model,
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = model,
            trigger = ("tokens", 550),
            keep = ("tokens", 200),
        ),
    ]
)

config = RunnableConfig({"configurable": {"thread_id": "test-1"}})

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [10]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content = f"Find hotels in {city}")]},
        config = config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~72 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='97a417b1-e1b5-4973-bb67-8107722fbaa4'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-08-27T16:32:40.0587805Z', 'done': True, 'done_reason': 'stop', 'total_duration': 807279600, 'load_duration': 9885600, 'prompt_eval_count': 160, 'prompt_eval_duration': 415417000, 'eval_count': 18, 'eval_duration': 351492000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--01a04410-fd52-74d3-b90f-fdc853dce38a-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'b7518a05-1df3-4c9f-a8cc-5a0d3980c21c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 160, 'output_tokens': 18, 'total_tokens': 178}), ToolMessage(content='Hotels in Paris:\n    1. Grand Hotel - 5 star, $350/night, spa, pool, gym\n    2. City Inn - 4 star, $180/night, bus

### Fraction

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware.summarization import ContextFraction

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model = model,
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = model,
            trigger = ("fraction", 0.005),  # 0.5% = ~640 tokens
            keep = ("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config: RunnableConfig = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config = config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~132 tokens (0.1031%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='43a03e59-f89f-4c81-9760-5326cc489adf'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-08-27T16:41:36.2756715Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3380711200, 'load_duration': 2892043800, 'prompt_eval_count': 151, 'prompt_eval_duration': 122176000, 'eval_count': 18, 'eval_duration': 358184000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--01a04419-21dd-7d81-8671-c313667e9560-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'a28fa6a6-8dda-4ce4-9eee-39e5a0a15fec', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'output_tokens': 18, 'total_tokens': 169}), ToolMessage(content='Hotels in Paris: Grand Hotel $350, City Inn $180, Budget Stay $75', name='search_hotels', id='96e0e118-fd56-4e

### Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [ ]:
agent = create_agent(
    model = model,
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool": {
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool": False,

            }
        )
    ]
)

In [ ]:
config: RunnableConfig = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    input = {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config = config
)

In [ ]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='51c5061d-0e5d-4129-8d36-c43881397340'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-08-27T16:43:57.3922092Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1043571400, 'load_duration': 8677400, 'prompt_eval_count': 233, 'prompt_eval_duration': 276579000, 'eval_count': 34, 'eval_duration': 684122000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--01a0441b-523d-7451-9126-8236dd0774d0-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'body': 'How are you?', 'subject': 'Hello'}, 'id': 'abab1df1-2141-489d-8c53-6012e0ceee09', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 233, 'output_tokens': 34, 'total_tokens': 267})],
 '__interrupt__': [Interrupt(value={'ac

In [ ]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config = config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been sent to the specified recipient with the specified subject and body.


In [18]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='df7c94ec-7c97-4b68-a7ef-4a1eb4b8ab5e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 97, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ff5f7093b3', 'id': 'chatcmpl-Co4Dem4tAOCx3wFsy4EMA623LU7ZP', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--7b23a6a7-07d9-4ad2-9724-359f1568186a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_A8s4sBnakCq3LfGK

### Reject

In [21]:
config: RunnableConfig = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config = config)

In [22]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config = config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Unfortunately, the user rejected the email tool call. I won't proceed with sending the email to John.


In [23]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='73613fd8-f7c0-4b0c-b2d7-3ba585675a77'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-08-27T16:46:58.7606965Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1142077000, 'load_duration': 8846200, 'prompt_eval_count': 233, 'prompt_eval_duration': 311805000, 'eval_count': 35, 'eval_duration': 771488000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--01a0441e-1651-7092-ba2d-12a1e6c4b86b-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'f1933087-67c5-4480-bab3-2199e8c16eef', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 233, 'output_tokens': 35, 'total_tokens': 268}),
  ToolMessage(content='User rejected the

### Editing

In [24]:
config: RunnableConfig = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    input = {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config = config
)

In [25]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='078c852f-cc67-4e0e-bee2-1e80e8d52598'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 95, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ff5f7093b3', 'id': 'chatcmpl-Co4I843MzNcaHARohEYFcE1RGYiJy', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--16d13698-ce64-4c47-aa2d-d1074c4b9032-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'wrong@email.com', 'subject': 'Test', 'body': 'Hello'}, 'id': 'call_8bku3hoUOJqnnR2FUM2ZO61d', '

In [25]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config = config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: The tool call response was used to format an answer to the original user question.


In [26]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='ad5bef0a-2b75-44dd-9e81-50b6c6e30a82'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-08-27T16:51:25.363157Z', 'done': True, 'done_reason': 'stop', 'total_duration': 972020000, 'load_duration': 9567200, 'prompt_eval_count': 231, 'prompt_eval_duration': 262886000, 'eval_count': 32, 'eval_duration': 643086000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--01a04422-2865-7591-a5c3-efdc624d6e0a-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'correct@email.com', 'subject': 'Corrected Subject', 'body': 'This was edited by human before sending'}, 'id': '4cc460d0-0e23-475b-b73c-7407bf319127'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 231, 'output_tokens': 32, 'total_tokens': 263}),
  Too